# 하방 변동성 사이징 — 최종본여러 시도 끝에 **재현되는 것 하나만** 남긴 노트북이다.## 여기까지의 경과| 시도 | 결과 ||---|---|| LSTM 가격 예측 | naive(전일가)에 패배 || LSTM 방향 예측 | 50~52%, p=0.41 — 동전던지기와 구분 안 됨 || 방향 백테스트 | 7개 전략 전부 Buy & Hold 에 패배 || LSTM 변동성 예측 | EWMA 한 줄에 패배 || **전체** 변동성 사이징 | 고정레버 대조군을 못 이김 || **하방** 변동성 사이징 | **처음으로 재현되는 개선** ← 이것만 남김 |**LSTM은 최종본에서 완전히 뺐다.** 네 번 시도해서 네 번 다 단순한 방법에 졌기 때문이다.남은 것은 `rolling(30)` 한 줄이다.---## 핵심 아이디어전체 변동성으로 포지션을 줄이면 **급락과 급등을 같이 피한다.**암호화폐 변동성은 상승/하락에 대체로 대칭이라 이 상쇄가 커서 Sharpe가 개선되지 않았다.**하방 변동성**은 음수 수익률만 본다.```pythonneg = r.clip(upper=0)                          # 음수만 남김dvol = np.sqrt((neg**2).rolling(30).mean())    # 하방 변동성```급등은 페널티를 주지 않고 급락 위험만 반영한다. 그래서 상승을 놓치지 않는다.---## 실측 결과 (평균 레버리지 0.85 로 통일, 수수료 편도 0.05%)### BTC/KRW 전체 (2021-03-31 ~ 2026-09-01, 1981일)| 전략 | 총수익 | CAGR | 연변동성 | MDD | Sharpe | Sortino ||---|---:|---:|---:|---:|---:|---:|| Buy & Hold | 58.8% | 8.9% | 54.0% | -74.3% | 0.43 | 0.60 || 고정레버 0.85 (대조군) | 64.0% | 9.5% | 45.9% | -67.6% | 0.43 | 0.60 || 전체 변동성 30일 | 87.6% | 12.3% | 43.8% | -67.4% | 0.48 | 0.68 || **하방 변동성 30일** | **113.7%** | **15.0%** | 43.5% | **-66.8%** | **0.54** | **0.77** |대조군 대비 — 하방 **Sharpe +0.111 / Sortino +0.172**, 전체 +0.056 / +0.083### ETH/KRW — 외부 검증 (학습에 쓰지 않은 자산)| 전략 | 총수익 | CAGR | MDD | Sharpe | Sortino ||---|---:|---:|---:|---:|---:|| Buy & Hold | 50.8% | 7.9% | -77.8% | 0.47 | 0.67 || 고정레버 0.85 | 70.9% | 10.4% | -71.4% | 0.47 | 0.67 || 전체 변동성 30일 | 80.1% | 11.4% | -72.0% | 0.48 | 0.69 || **하방 변동성 30일** | **204.2%** | **22.8%** | **-66.2%** | **0.64** | **0.94** |### BTC/KRW 후반 50% — 기간 검증| 전략 | 총수익 | CAGR | MDD | Sharpe | Sortino ||---|---:|---:|---:|---:|---:|| Buy & Hold | 92.5% | 27.3% | -49.5% | 0.75 | 1.15 || 고정레버 0.85 | 81.2% | 24.5% | -43.4% | 0.75 | 1.15 || 전체 변동성 30일 | 74.3% | 22.7% | -47.5% | 0.72 | 1.09 || **하방 변동성 30일** | **87.0%** | **25.9%** | **-45.1%** | **0.79** | **1.21** |세 검증 모두에서 대조군을 이겼다. 다만 **후반 구간 개선폭은 +0.04 로 작다.**그리고 후반 구간에서는 Buy & Hold 의 총수익(92.5%)을 넘지 못한다 —위험 대비 비율이 나아진 것이지 더 번 것이 아니다.---## 실전 프로토콜 검증 — 스케일을 walk-forward 로 재추정위 표들은 **전체 구간으로 스케일을 정했다.** 미래를 조금 본 셈이다.실전에서는 매 시점 과거 데이터로만 스케일을 다시 잡아야 한다(1년 burn-in, 30일마다 재추정).바꿔서 다시 돌린 결과다.| 검증 | 대조군 대비 (in-sample) | 대조군 대비 (walk-forward) | 차이 ||---|---:|---:|---:|| BTC 전체 | +0.077 | **+0.040** | -0.037 || ETH | +0.125 | **+0.098** | -0.027 || BTC 후반 | +0.025 | **+0.043** | +0.017 |**우위가 살아남는다.** BTC 전체에서는 개선폭이 절반으로 깎이고,ETH에서는 대부분 유지되며, 후반 구간에서는 오히려 조금 늘었다.평균적으로 약 30% 정도 깎인다고 보면 된다.### walk-forward 기준 상세 (BTC 전체, 2021-10-10 ~ 2026-08-31, 1787일)| 전략 | 총수익 | CAGR | MDD | Sharpe | Sortino | 평균레버 ||---|---:|---:|---:|---:|---:|---:|| Buy & Hold | 62.2% | 10.4% | -74.3% | 0.45 | 0.64 | 1.00 || 고정레버 0.94 (대조군) | 63.2% | 10.5% | -71.7% | 0.45 | 0.64 | 0.94 || 전체 변동성 (walk-forward) | 69.5% | 11.4% | -72.7% | 0.46 | 0.65 | 0.95 || **하방 변동성 (walk-forward)** | **79.0%** | **12.6%** | -72.8% | **0.49** | **0.69** | 0.94 |### 국면별로 뜯어보면 — 우위는 상승장에서 나온다전체 숫자만 보면 ETH 는 Buy & Hold -20.4% vs 확정판 +20.3% 로 극적이다.하지만 **국면을 쪼개면 이야기가 달라진다.**| 국면 | 일수 | B&H | 대조군 | 확정판 | 판정 ||---|---:|---:|---:|---:|:--|| BTC 2022 폭락 | 417 | -72.8% | -67.4% | **-71.2%** | **패** || BTC 2023-24 상승 | 731 | 563.6% | 532.3% | **562.7%** | 승 || BTC 2025-26 하락 | 332 | -39.9% | -37.1% | **-37.4%** | **패** || ETH 2022 폭락 | 417 | -72.4% | -66.0% | **-68.9%** | **패** || ETH 2023-24 상승 | 731 | 226.3% | 218.4% | **242.9%** | 승 || ETH 2025-26 하락 | 332 | -50.7% | -45.8% | **-47.6%** | **패** |**하락 국면 4곳에서 전부 대조군에 진다.**전체 우위는 2023-24 상승장 한 구간에서 나온 것이다.왜인가: 하방변동성은 **이미 떨어진 뒤에** 포지션을 줄인다.그리고 폭락 후 변동성이 높게 유지되는 동안 나오는 급반등을 놓친다.하락장은 상시 고변동이라 사실상 고정레버와 같아지고, 반등 미참여분만 손해다.반대로 상승장에서는 조정 구간에만 선택적으로 줄이고잔잔한 랠리에는 풀로 실어서 이득이 난다.> **이것은 낙폭 방어 전략이 아니라 '상승장 안에서 조정을 피하는' 전략이다.**> 지금처럼 고점 대비 -40% 인 국면에서는 고정레버 0.85 와 실질적으로 같다.### burn-in / 재추정주기 민감도walk-forward 에 임의로 넣은 두 값(burn-in 252일, 재추정 30일)도 흔들어봤다.burn-in 4가지 × 재추정 7가지 = 28조합 × 세 검증 = **84칸 전부 양수**다.| 검증 | 최소 | 최대 | 중앙값 ||---|---:|---:|---:|| BTC 전체 | +0.058 | +0.062 | **+0.060** || ETH | +0.096 | +0.106 | **+0.104** || BTC 후반 | +0.051 | +0.054 | **+0.053** |**두 파라미터는 결과에 거의 영향이 없다.** 폭이 0.003~0.010 에 불과하다.스케일은 평균 노출도만 정하고 타이밍은 하방변동성 자체에서 나오는데,대조군을 같은 평균 노출도로 맞췄으므로 스케일의 영향이 상쇄되기 때문이다.창 길이(25~40 고원)보다도 훨씬 안정적이다.**결국 이 전략에서 실제로 민감한 파라미터는 창 길이 하나뿐이다.**---## 왜 이번엔 믿을 만한가 — 검증 장치 3개앞선 시도들이 실패한 이유는 대조군 없이 숫자만 봤기 때문이다. 이번엔 세 겹으로 막았다.**① 평균 레버리지 통일**방법마다 평균 노출도가 다르면 상승장에서는 노출 높은 쪽이 그냥 이긴다.모든 방법을 평균 레버리지 0.85로 맞춰서, **타이밍 능력만** 비교되게 했다.**② 고정레버 대조군**노출도만 0.85로 낮추고 타이밍은 전혀 없는 전략.이걸 못 이기면 "변동성 타이밍"에 아무 가치가 없다는 뜻이다.앞서 전체 변동성 사이징이 바로 여기서 걸렸다.**③ 창 길이 고원(plateau) 확인**30일 하나만 좋으면 그 창을 골라낸 것(과적합)이다.25/30/35/40이 **연속으로** 세 검증 모두에서 양수여야 진짜 신호다.

# 0. 준비

In [ ]:
!pip install -q finance-datareader

In [ ]:
import numpy as np, pandas as pdimport matplotlib.pyplot as pltimport FinanceDataReader as fdrimport warnings; warnings.filterwarnings('ignore')FEE = 0.0005          # 편도 0.05% (왕복 0.1%)MAXLEV = 1.0          # 축소만, 확대 없음 (스윕으로 확정)EXPOSURE = 0.85       # 모든 방법의 평균 레버리지를 이 값으로 통일WINDOW = 30           # 하방 변동성 창

---# 1. 데이터와 하방 변동성`shift(1)` 이 중요하다. t일의 포지션은 **t-1일까지의 정보로만** 결정해야 한다.이걸 빼면 미래를 보고 매매하는 셈이 되어 결과가 환상적으로 좋아진다.

In [ ]:
def load_coin(ticker, start='2021-01-01', end=None):    df = fdr.DataReader(ticker, start, end)    df = df.dropna(subset=['Open','High','Low','Close'])     # 당일 미확정 행 제거    return df[['Open','High','Low','Close','Volume']]def downside_vol(close, window=WINDOW):    """하방 변동성: 음수 수익률만 사용"""    r = np.log(close/close.shift(1))    neg = r.clip(upper=0)                                     # 양수는 0 으로    return np.sqrt((neg**2).rolling(window).mean()).shift(1)  # shift(1) = 미래 차단def total_vol(close, window=WINDOW):    r = np.log(close/close.shift(1))    return r.rolling(window).std().shift(1)btc = load_coin('BTC/KRW')r = np.log(btc.Close/btc.Close.shift(1))print(f"{len(btc)}행  {btc.index[0].date()} ~ {btc.index[-1].date()}")print(f"일간 변동성 {r.std():.2%},  상승일 {(r>0).mean():.1%}")dv, tv = downside_vol(btc.Close), total_vol(btc.Close)print(f"\n하방 변동성 평균 {dv.mean():.4f}  /  전체 변동성 평균 {tv.mean():.4f}")print(f"두 지표의 상관 {dv.corr(tv):.3f}")print("상관이 높지만 1은 아니다. 그 차이가 이 전략의 근거다.")

---# 2. 공정 비교 장치**평균 레버리지 통일**이 이 노트북의 핵심이다.`1/변동성` 을 그대로 쓰면 방법마다 평균 노출도가 달라지고,상승장에서는 노출도가 높은 쪽이 실력과 무관하게 이긴다.이분법으로 스케일을 찾아 모든 방법의 평균 레버리지를 정확히 같게 맞춘다.

In [ ]:
def calibrate(inv_vol, target_exposure=EXPOSURE, maxlev=MAXLEV):    """clip 후 평균 레버리지가 target_exposure 가 되도록 스케일 탐색"""    lo, hi = 1e-8, 1e8    for _ in range(200):        mid = np.sqrt(lo*hi)        if np.clip(mid*inv_vol, 0, maxlev).mean() < target_exposure:            lo = mid        else:            hi = mid    return np.clip(np.sqrt(lo*hi)*inv_vol, 0, maxlev)def stats(ret, name, lev=None):    eq = np.cumprod(1+ret); peak = np.maximum.accumulate(eq)    yrs = len(ret)/365    dn = ret[ret < 0]    return dict(전략=name, 총수익=eq[-1]-1,                CAGR=eq[-1]**(1/yrs)-1 if eq[-1] > 0 else -1.0,                연변동성=ret.std()*np.sqrt(365),                MDD=((eq-peak)/peak).min(),                Sharpe=ret.mean()/(ret.std()+1e-12)*np.sqrt(365),                Sortino=ret.mean()/(dn.std()+1e-12)*np.sqrt(365),                평균레버=lev.mean() if lev is not None else 1.0)def backtest(ticker, frac=0.0, burn=90, label=''):    df = load_coin(ticker)    c = df.Close    d = pd.DataFrame({'dvol': downside_vol(c), 'tvol': total_vol(c),                      'nxt': np.log(c/c.shift(1)).shift(-1)}).iloc[burn:].dropna()    if frac: d = d.iloc[int(len(d)*frac):]    simple = np.exp(d['nxt'].to_numpy())-1    rows = [stats(simple, 'Buy & Hold')]    flat = np.full(len(simple), EXPOSURE)    rows.append(stats(flat*simple, f'고정레버 {EXPOSURE} (대조군)', flat))    for key, nm in [('tvol', f'전체 변동성 {WINDOW}일'), ('dvol', f'하방 변동성 {WINDOW}일')]:        lev = calibrate(1.0/np.clip(d[key].to_numpy(), 1e-6, None))        cost = FEE*np.abs(np.diff(np.concatenate([[0.], lev])))        rows.append(stats(lev*simple - cost, nm, lev))    T = pd.DataFrame(rows)    f = T.copy()    for x in ['총수익','CAGR','연변동성','MDD']: f[x] = f[x].map(lambda v: f"{v:.1%}")    for x in ['Sharpe','Sortino','평균레버']: f[x] = f[x].map(lambda v: f"{v:.2f}")    print(f"\n{'='*100}")    print(f"{label}   n={len(simple)}   {d.index[0].date()} ~ {d.index[-1].date()}")    print('='*100)    print(f.to_string(index=False))    return T, d, simple

## 검증 1 — BTC 전체 구간

In [ ]:
T_btc, d_btc, s_btc = backtest('BTC/KRW', label='[BTC/KRW 전체]')ctl = T_btc.iloc[1]; dn = T_btc.iloc[3]; tv_ = T_btc.iloc[2]print(f"\n하방 - 대조군 : Sharpe {dn.Sharpe-ctl.Sharpe:+.3f}, Sortino {dn.Sortino-ctl.Sortino:+.3f}")print(f"전체 - 대조군 : Sharpe {tv_.Sharpe-ctl.Sharpe:+.3f}, Sortino {tv_.Sortino-ctl.Sortino:+.3f}")

## 검증 2 — ETH (외부 자산)BTC에서 통한 게 우연이 아니라면 ETH에서도 나타나야 한다.

In [ ]:
T_eth, _, _ = backtest('ETH/KRW', label='[ETH/KRW 외부 검증]')

## 검증 3 — BTC 후반 50% (다른 기간)

In [ ]:
T_half, _, _ = backtest('BTC/KRW', frac=0.5, label='[BTC/KRW 후반 50%]')

---# 3. 창 길이 고원 확인 — 30일을 골라낸 것인가?**이 셀이 이 노트북에서 가장 중요하다.**특정 창에서만 좋으면 그 창을 고른 것이다(과적합).25/30/35/40 처럼 **인접한 값들이 연속으로** 좋아야 진짜 신호다.주의: 창마다 `rolling` 시작점이 달라지므로 **동일 표본(burn=90)** 으로 맞춰야 한다.이걸 안 하면 창 90은 90일 늦게 시작해서 비교가 오염된다.

In [ ]:
def improvement(ticker, window, kind='down', frac=0.0, burn=90):    df = load_coin(ticker); c = df.Close    pv = downside_vol(c, window) if kind == 'down' else total_vol(c, window)    d = pd.DataFrame({'pv': pv, 'nxt': np.log(c/c.shift(1)).shift(-1)}).iloc[burn:].dropna()    if frac: d = d.iloc[int(len(d)*frac):]    simple = np.exp(d['nxt'].to_numpy())-1    lev = calibrate(1.0/np.clip(d['pv'].to_numpy(), 1e-6, None))    cost = FEE*np.abs(np.diff(np.concatenate([[0.], lev])))    s = stats(lev*simple-cost, '', lev)    ctl = stats(np.full(len(simple), EXPOSURE)*simple, '')    return s['Sharpe']-ctl['Sharpe']WINDOWS = [10, 15, 20, 25, 30, 35, 40, 50, 60, 90]cases = [('BTC/KRW', 0.0, 'BTC 전체'), ('ETH/KRW', 0.0, 'ETH'), ('BTC/KRW', 0.5, 'BTC 후반')]for kind, klab in [('down', '하방 변동성'), ('total', '전체 변동성')]:    print(f"\n{'='*76}")    print(f"{klab} — 고정레버 대조군 대비 Sharpe 개선폭")    print('='*76)    print(f"{'창':>4s}" + ''.join(f"{l:>20s}" for _,_,l in cases) + f"{'모두+':>8s}")    for w in WINDOWS:        ds = [improvement(tk, w, kind, fr) for tk, fr, _ in cases]        print(f"{w:4d}" + ''.join(f"{x:>20.3f}" for x in ds)              + f"{('  O' if all(x>0 for x in ds) else '  .'):>8s}")print()print("하방: 25/30/35/40 이 연속으로 세 검증 모두 양수 -> 고원. 신호로 볼 수 있다.")print("전체: 세 검증 모두 양수인 창이 하나도 없다 -> 효과 없음.")

---# 3.5 실전 프로토콜 — 스케일을 walk-forward 로 재추정지금까지의 `calibrate()` 는 **전체 구간**으로 스케일을 정했다."평균 레버리지가 0.85가 되도록" 맞추려면 전 구간의 변동성 분포를 알아야 하는데,2021년에 그걸 알 수는 없다. 미래를 조금 본 셈이다.실전 규칙으로 바꾼다.- **burn-in 252일** (최소 1년 쌓인 뒤부터 매매 시작)- **30일마다** 그때까지의 과거 데이터로만 스케일 재추정이렇게 하면 수치가 얼마나 깎이는지 본다.

In [ ]:
BURN, REBAL = 252, 30def scale_for(inv, target=EXPOSURE, maxlev=MAXLEV):    lo, hi = 1e-8, 1e8    for _ in range(100):        mid = np.sqrt(lo*hi)        if np.clip(mid*inv, 0, maxlev).mean() < target: lo = mid        else: hi = mid    return np.sqrt(lo*hi)def lev_walkforward(inv):    """매 시점 과거 데이터로만 스케일 재추정"""    n = len(inv); lev = np.full(n, np.nan); sc = None    for t in range(n):        if t < BURN: continue        if sc is None or (t-BURN) % REBAL == 0:            sc = scale_for(inv[:t])          # t 시점까지만 사용        lev[t] = min(sc*inv[t], MAXLEV)    return levdef compare_wf(ticker, frac=0.0, label=''):    c = load_coin(ticker).Close    d = pd.DataFrame({'dv': downside_vol(c), 'tv': total_vol(c),                      'nxt': np.log(c/c.shift(1)).shift(-1)}).dropna()    inv_d = 1.0/np.clip(d['dv'].to_numpy(), 1e-6, None)    inv_t = 1.0/np.clip(d['tv'].to_numpy(), 1e-6, None)    L_in = np.clip(scale_for(inv_d)*inv_d, 0, MAXLEV)    L_wf = lev_walkforward(inv_d)    T_wf = lev_walkforward(inv_t)    ok = ~np.isnan(L_wf)    if frac:        idx = np.where(ok)[0]; ok[:] = False; ok[idx[int(len(idx)*frac):]] = True    simple = np.exp(d['nxt'].to_numpy()[ok])-1    def net(lv):        lv = lv[ok]        return lv*simple - FEE*np.abs(np.diff(np.concatenate([[0.], lv]))), lv    exp_wf = float(np.mean(L_wf[ok]))            # 대조군을 실제 평균노출에 맞춤    flat = np.full(len(simple), exp_wf)    rows = [stats(simple, 'Buy & Hold'),            stats(flat*simple, f'고정레버 {exp_wf:.2f} (대조군)', flat)]    for nm, lv in [('전체 변동성 (walk-forward)', T_wf),                   ('하방 변동성 (in-sample 스케일)', L_in),                   ('하방 변동성 (walk-forward)', L_wf)]:        rr, lvv = net(lv); rows.append(stats(rr, nm, lvv))    T = pd.DataFrame(rows)    f = T.copy()    for x in ['총수익','CAGR','연변동성','MDD']: f[x] = f[x].map(lambda v: f"{v:.1%}")    for x in ['Sharpe','Sortino','평균레버']: f[x] = f[x].map(lambda v: f"{v:.2f}")    print(f"\n{'='*100}")    print(f"{label}   n={int(ok.sum())}   {d.index[ok][0].date()} ~ {d.index[ok][-1].date()}")    print('='*100)    print(f.to_string(index=False))    ctl = T.iloc[1]    print(f"  대조군 대비:  in-sample {T.iloc[3].Sharpe-ctl.Sharpe:+.3f}"          f"   walk-forward {T.iloc[4].Sharpe-ctl.Sharpe:+.3f}"          f"   (차이 {T.iloc[4].Sharpe-T.iloc[3].Sharpe:+.3f})")    return Tcompare_wf('BTC/KRW', label='[BTC/KRW 전체]')compare_wf('ETH/KRW', label='[ETH/KRW -- 이 구간은 하락장이다]')compare_wf('BTC/KRW', frac=0.5, label='[BTC/KRW 후반 50%]')print()print("세 검증 모두 walk-forward 에서도 대조군을 이긴다.")print("평균적으로 개선폭이 약 30% 깎이지만 부호는 유지된다.")

---# 3.6 burn-in / 재추정주기 민감도walk-forward 에는 임의로 정한 값이 둘 있다 — burn-in 252일, 재추정 30일.이 둘을 바꿔도 결과가 유지되는지 확인한다. 창 길이 때와 같은 원칙이다.burn-in 4가지 × 재추정주기 7가지 = 28조합을, 세 검증에서 각각 돌린다.**모든 조합이 동일 표본(burn 504일 시점부터)에서 평가되도록** 맞춘다.

In [ ]:
BURNS  = [126, 252, 378, 504]          # 0.5 / 1 / 1.5 / 2년REBALS = [1, 5, 10, 20, 30, 60, 120]COMMON = max(BURNS)                    # 동일 표본 기준점def sens(ticker, burn, rebal, frac=0.0):    c = load_coin(ticker).Close    d = pd.DataFrame({'dv': downside_vol(c),                      'nxt': np.log(c/c.shift(1)).shift(-1)}).dropna()    inv = 1.0/np.clip(d['dv'].to_numpy(), 1e-6, None)    n = len(inv); L = np.full(n, np.nan); sc = None    for t in range(n):        if t < burn: continue        if sc is None or (t-burn) % rebal == 0:            sc = scale_for(inv[:t])        L[t] = min(sc*inv[t], MAXLEV)    ok = np.zeros(n, bool); ok[COMMON:] = True    if frac:        idx = np.where(ok)[0]; ok[:] = False        ok[idx[int(len(idx)*frac):]] = True    simple = np.exp(d['nxt'].to_numpy()[ok])-1    lv = L[ok]    net = lv*simple - FEE*np.abs(np.diff(np.concatenate([[0.], lv])))    ctl = np.full(len(simple), lv.mean())*simple      # 같은 평균노출 대조군    sh = lambda x: x.mean()/(x.std()+1e-12)*np.sqrt(365)    return sh(net) - sh(ctl)for tk, fr, lab in [('BTC/KRW', 0.0, 'BTC 전체'),                    ('ETH/KRW', 0.0, 'ETH'),                    ('BTC/KRW', 0.5, 'BTC 후반')]:    print()    print('='*78)    print(f'[{lab}]  대조군 대비 Sharpe 개선폭')    print('='*78)    print('burn/rebal'.rjust(12) + ''.join(f'{r:>9d}' for r in REBALS))    g = []    for b in BURNS:        row = [sens(tk, b, rb, fr) for rb in REBALS]        g.append(row)        print(f'{b:>5d}({b/252:.1f}년)' + ''.join(f'{x:>9.3f}' for x in row))    g = np.array(g)    print(f'  {g.size}칸 중 양수 {int((g>0).sum())}칸   '          f'최소 {g.min():+.3f}  최대 {g.max():+.3f}  중앙값 {np.median(g):+.3f}')print()print('두 파라미터 모두 결과에 거의 영향이 없다.')print('스케일은 평균 노출도만 정하고, 타이밍은 하방변동성 자체에서 나오기 때문이다.')print('대조군을 같은 평균 노출도로 맞췄으므로 스케일 효과는 상쇄된다.')

---# 3.7 국면별 성적 — 우위는 어디서 나오는가전체 Sharpe 개선이 **모든 국면에 고르게 분포하는지** 확인한다.한 구간에 몰려 있다면 그 구간이 재현되지 않을 때 전략도 작동하지 않는다.

In [ ]:
REGIMES = [('2022 폭락',    '2021-11-10', '2022-12-31'),           ('2023-24 상승', '2023-01-01', '2024-12-31'),           ('2025-26 하락', '2025-10-06', '2030-01-01'),           ('전체',         '2000-01-01', '2030-01-01')]def regime_table(ticker):    c = load_coin(ticker).Close    d = pd.DataFrame({'dv': downside_vol(c),                      'nxt': np.log(c/c.shift(1)).shift(-1)}).dropna()    inv = 1.0/np.clip(d['dv'].to_numpy(), 1e-6, None)    lev = lev_walkforward(inv)    base = ~np.isnan(lev)    def tot(x): return np.cumprod(1+x)[-1]-1    def mdd(x):        eq = np.cumprod(1+x); pk = np.maximum.accumulate(eq)        return ((eq-pk)/pk).min()    rows = []    for nm, s_, e_ in REGIMES:        m = base & (d.index >= s_) & (d.index <= e_)        if m.sum() < 30: continue        simple = np.exp(d['nxt'].to_numpy()[m])-1        lv = lev[m]        net = lv*simple - FEE*np.abs(np.diff(np.concatenate([[0.], lv])))        ctl = np.full(len(simple), lv.mean())*simple        rows.append(dict(국면=nm, 일수=int(m.sum()), BH=tot(simple),                         대조군=tot(ctl), 확정판=tot(net),                         판정='승' if tot(net) > tot(ctl) else '패',                         확정_MDD=mdd(net), 평균레버=lv.mean()))    return pd.DataFrame(rows)for tk in ['BTC/KRW', 'ETH/KRW']:    R = regime_table(tk)    f = R.copy()    for x in ['BH','대조군','확정판','확정_MDD']: f[x] = f[x].map(lambda v: f"{v:.1%}")    f['평균레버'] = f['평균레버'].map(lambda v: f"{v:.2f}")    print()    print(f'[{tk}]')    print(f.to_string(index=False))print()print('하락 국면에서는 확정판이 고정레버 대조군에 진다.')print('전체 우위는 상승장 구간에서 나온 것이다 -- 낙폭 방어 전략이 아니다.')print('하방변동성은 이미 떨어진 뒤 포지션을 줄이고, 이후 급반등을 놓치기 때문이다.')

---# 4. 자산배분 곡선과 레버리지

In [ ]:
lev = calibrate(1.0/np.clip(d_btc['dvol'].to_numpy(), 1e-6, None))cost = FEE*np.abs(np.diff(np.concatenate([[0.], lev])))ret_dn = lev*s_btc - costflat = np.full(len(s_btc), EXPOSURE)fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True,                       gridspec_kw={'height_ratios': [2, 1]})ax[0].plot(d_btc.index, np.cumprod(1+s_btc), label='Buy & Hold', lw=1.2)ax[0].plot(d_btc.index, np.cumprod(1+flat*s_btc), label=f'고정레버 {EXPOSURE} (대조군)', lw=1.2)ax[0].plot(d_btc.index, np.cumprod(1+ret_dn), label=f'하방 변동성 {WINDOW}일', lw=1.6)ax[0].set_yscale('log'); ax[0].set_title('자산 곡선 (로그 스케일)')ax[0].legend(); ax[0].grid(alpha=.3)ax[1].plot(d_btc.index, lev, lw=.9, color='tab:green')ax[1].axhline(EXPOSURE, ls='--', c='gray', lw=1, label=f'평균 {EXPOSURE}')ax[1].set_title('레버리지 (하방 변동성이 커지면 자동으로 축소)')ax[1].legend(); ax[1].grid(alpha=.3)plt.tight_layout(); plt.show()

---# 5. 오늘의 포지션실제로 쓴다면 이 값이다.

In [ ]:
c = btc.Closedv_now = downside_vol(c, WINDOW)d_all = pd.DataFrame({'dvol': dv_now, 'nxt': np.log(c/c.shift(1)).shift(-1)}).iloc[90:]hist = d_all.dropna()lev_hist = calibrate(1.0/np.clip(hist['dvol'].to_numpy(), 1e-6, None))# 과거에서 구한 스케일을 현재 하방변동성에 적용scale = (lev_hist/np.clip(1.0/np.clip(hist['dvol'].to_numpy(), 1e-6, None), 1e-12, None))scale = np.median(scale[lev_hist < MAXLEV]) if (lev_hist < MAXLEV).any() else 1.0latest_dvol = float(dv_now.dropna().iloc[-1])pos = float(np.clip(scale/latest_dvol, 0, MAXLEV))print(f"기준일        : {c.index[-1].date()}")print(f"하방 변동성    : {latest_dvol:.4f}  (과거 중앙값 {float(np.nanmedian(dv_now)):.4f})")print(f"권장 포지션    : {pos:.2f}  (0=전량현금, 1.0=풀매수)")print()print("변동성이 평소보다 크면 1.0 미만으로 줄어든다.")print("이 값은 방향 예측이 아니라 '얼마나 실을지' 만 말해준다.")

---# 정리## 남긴 것```pythonneg = r.clip(upper=0)dvol = np.sqrt((neg**2).rolling(30).mean()).shift(1)position = clip(scale/dvol, 0, 1.0)```이게 전부다. LSTM도, 기술지표 23개도, seed 앙상블도 없다.**네 번의 실패 끝에 남은 건 세 줄이다.**## 왜 이것만 살아남았나| | 방향 예측 | 전체 변동성 | 하방 변동성 ||---|---|---|---|| 대조군 대비 | 못 이김 | 못 이김 | **이김** || 외부 자산(ETH) | 재현 안 됨 | 재현 안 됨 | **재현됨** || 다른 기간 | 재현 안 됨 | 음수 | **재현됨** || 파라미터 민감도 | — | 고원 없음 | **25~40 고원** |## 그래도 남는 한계 — 반드시 읽을 것**1. 개선폭이 작다.** BTC 후반 구간에서는 Sharpe +0.04 다.표본이 다르면 사라질 수 있는 크기다. 전체 구간(+0.11)과 ETH(+0.17)에서는 더 크지만,가장 최근 구간에서 가장 작다는 점이 걸린다.**2. 국면 커버리지는 충분하지만, 우위가 국면에 몰려 있다.**검증 구간(2021~2026)은 2022 폭락(BTC -63.2%, ETH -66.3%)과2025-26 하락(고점 대비 BTC -40%, ETH -51%)을 모두 포함한다.사이클 커버리지 자체는 문제없다.문제는 **하락 국면 4곳에서 전부 대조군에 진다**는 것이다(위 국면별 표).전체 Sharpe 개선은 상승장 한 구간에서 나온 것이라,하락 국면에 투입하면 고정레버와 다르지 않다.**3. 자산 2개, 기간 5년이다.** 통계적으로 충분하지 않다.최소한 여러 자산군과 여러 사이클에서 재검증해야 한다.**4. 수익을 늘리는 전략이 아니다.** 같은 위험에서 조금 나은 비율을 얻는 것이지,방향을 맞히는 게 아니다. 여전히 하락장에서는 돈을 잃는다.**5. 이 노트북 자체도 여러 변형을 시도한 끝에 나왔다.**9개 방법 × 10개 창 × 3개 검증을 봤다면, 그중 하나가 우연히 좋아 보일 확률도 있다.고원 확인과 외부 검증으로 방어했지만 **완전히 배제할 수는 없다.**## 다음에 한다면1. **더 긴 기간** — 2017~2018 하락장 포함해서 재검증2. **여러 자산** — 코인 10종, 그리고 주식/원자재에서도 되는지3. **더 긴 재추정 주기 / 다른 목표 노출도(EXPOSURE)** — burn-in 과 재추정주기는   확인했다(84칸 전부 양수). EXPOSURE=0.85 라는 값 자체는 아직 안 흔들어봤다4. **하방 + 추세 결합 재검토** — 별도 실험에서 BTC 전체는 이쪽이 좋았으나   ETH에서는 하방 단독이 나았다. 파라미터가 3개라 과적합 위험이 커서 최종본에서 뺐다> 학습용 코드다. 실제 운용 전에 위 한계 5가지를 각각 해소해야 한다.